# Mobile Social Media PCAP to CSV Dataset Pipeline

This Colab notebook converts mobile social-media traffic captures from PCAP/PCAPNG into flow-based CSV files using NFStream.

It also counts rows and merges multiple capture days for each application.

**Input folder:** `/content/drive/MyDrive/mobile`  
**Individual CSV output:** `/content/drive/MyDrive/mobile_output`  
**Merged app-level CSV output:** `/content/drive/MyDrive/mobile_merged_output`

Expected capture filename examples:

- `facebook_day1.pcap`
- `facebook_day2.pcap`
- `facebook_day3.pcap`
- `instagram_day1.pcapng`

The application label is extracted from the filename before `_day`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q nfstream pandas


In [ ]:
from pathlib import Path

# Input folder containing PCAP/PCAPNG files
INPUT_DIR = Path('/content/drive/MyDrive/mobile')

# Output folder for individual converted CSV files
OUTPUT_DIR = Path('/content/drive/MyDrive/mobile_output')

# Output folder for merged app-level CSV files
MERGED_DIR = Path('/content/drive/MyDrive/mobile_merged_output')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

print('Input folder:', INPUT_DIR)
print('Individual CSV output folder:', OUTPUT_DIR)
print('Merged CSV output folder:', MERGED_DIR)
print('Input folder exists:', INPUT_DIR.exists())


In [ ]:
import re

def extract_label_and_day(file_stem):
    """
    Extract app label and capture day from filenames like:
    facebook_day1, facebook_day2, instagram_day3.

    If '_day' is missing, the label becomes the part before the first underscore.
    """
    name = file_stem.lower().strip()

    match = re.search(r'(.+?)_day(\d+)', name)
    if match:
        label = match.group(1).strip('_- ')
        day = 'day' + match.group(2)
        return label, day

    parts = name.split('_')
    label = parts[0]
    day = 'unknown'
    return label, day


In [ ]:
import pandas as pd
from nfstream import NFStreamer

pcap_files = sorted(list(INPUT_DIR.glob('*.pcap')) + list(INPUT_DIR.glob('*.pcapng')))

print(f'Found {len(pcap_files)} capture files')

if len(pcap_files) == 0:
    print('No PCAP/PCAPNG files found. Check that your files are inside /content/drive/MyDrive/moblie')

for pcap_file in pcap_files:
    print('\n' + '=' * 80)
    print(f'Processing: {pcap_file.name}')

    capture_name = pcap_file.stem
    label, capture_day = extract_label_and_day(capture_name)

    print(f'Label: {label}')
    print(f'Capture day: {capture_day}')

    streamer = NFStreamer(
        source=str(pcap_file),
        statistical_analysis=True,
        splt_analysis=True,
        n_dissections=20
    )

    df = streamer.to_pandas()

    # Keep ALL NFStream extracted features, then add metadata columns.
    df['Label'] = label
    df['capture_day'] = capture_day
    df['capture_file'] = pcap_file.name

    # Move Label to the first column.
    cols = ['Label'] + [c for c in df.columns if c != 'Label']
    df = df[cols]

    output_csv = OUTPUT_DIR / f'{capture_name}.csv'
    df.to_csv(output_csv, index=False)

    print(f'Saved: {output_csv}')
    print(f'Rows: {len(df):,}')
    print(f'Columns: {len(df.columns):,}')

print('\nConversion completed.')


## Fast Row Count for Individual CSV Files

This cell counts rows without loading the full CSV files into memory.


In [ ]:
from pathlib import Path
import pandas as pd

csv_files = sorted([
    f for f in OUTPUT_DIR.glob('*.csv')
    if 'summary' not in f.name.lower()
])

print(f'CSV files found: {len(csv_files)}')

summary = []
grand_total = 0

if len(csv_files) == 0:
    print('No CSV files found in:', OUTPUT_DIR)
else:
    print('=' * 80)
    print('FAST ROW COUNT REPORT - INDIVIDUAL CSV FILES')
    print('=' * 80)

    for csv_file in csv_files:
        with open(csv_file, 'r', encoding='utf-8', errors='ignore') as f:
            row_count = sum(1 for _ in f) - 1

        row_count = max(row_count, 0)
        grand_total += row_count

        # Read only header for column count.
        column_count = len(pd.read_csv(csv_file, nrows=0).columns)

        summary.append({
            'file_name': csv_file.name,
            'rows': row_count,
            'columns': column_count
        })

        print(f'{csv_file.name:<45} {row_count:>12,} rows   {column_count:>5} columns')

    print('=' * 80)
    print(f'{"TOTAL ROWS":<45} {grand_total:>12,}')
    print('=' * 80)

    summary_df = pd.DataFrame(summary, columns=['file_name', 'rows', 'columns'])
    summary_df = pd.concat([
        summary_df,
        pd.DataFrame([{'file_name': 'TOTAL', 'rows': grand_total, 'columns': ''}])
    ], ignore_index=True)

    summary_path = OUTPUT_DIR / 'row_count_summary.csv'
    summary_df.to_csv(summary_path, index=False)
    print('Summary saved to:', summary_path)


## Merge 3 Captures Per Application

This cell groups files by application name and merges the daily captures into one CSV per application.

Example:

- `facebook_day1.csv`
- `facebook_day2.csv`
- `facebook_day3.csv`

becomes:

- `facebook.csv`


In [ ]:
from pathlib import Path
import pandas as pd

input_csv_files = sorted([
    f for f in OUTPUT_DIR.glob('*.csv')
    if 'summary' not in f.name.lower()
])

apps = {}

for csv_file in input_csv_files:
    app_name, _ = extract_label_and_day(csv_file.stem)
    apps.setdefault(app_name, []).append(csv_file)

print(f'Applications detected: {len(apps)}')
print('Applications:', ', '.join(sorted(apps.keys())))

grand_total_merged = 0
merge_summary = []

for app_name, files in sorted(apps.items()):
    print('\n' + '=' * 80)
    print(f'MERGING APPLICATION: {app_name.upper()}')
    print('=' * 80)

    dfs = []
    reference_columns = None
    total_rows_before_merge = 0

    for file in sorted(files):
        df = pd.read_csv(file)

        rows = len(df)
        total_rows_before_merge += rows
        print(f'{file.name:<45} {rows:>12,} rows')

        # Check schema consistency.
        current_columns = set(df.columns)
        if reference_columns is None:
            reference_columns = current_columns
        elif current_columns != reference_columns:
            missing = reference_columns - current_columns
            extra = current_columns - reference_columns
            print(f'WARNING: Column mismatch in {file.name}')
            if missing:
                print('  Missing columns:', sorted(missing))
            if extra:
                print('  Extra columns:', sorted(extra))

        # Ensure Label is correct after merging.
        df['Label'] = app_name

        # Keep Label first.
        cols = ['Label'] + [c for c in df.columns if c != 'Label']
        df = df[cols]

        dfs.append(df)

    merged_df = pd.concat(dfs, ignore_index=True)

    output_file = MERGED_DIR / f'{app_name}.csv'
    merged_df.to_csv(output_file, index=False)

    merged_rows = len(merged_df)
    grand_total_merged += merged_rows

    merge_summary.append({
        'application': app_name,
        'number_of_input_files': len(files),
        'merged_rows': merged_rows,
        'columns': len(merged_df.columns),
        'output_file': output_file.name
    })

    print('-' * 80)
    print(f'Merged files: {len(files)}')
    print(f'Merged rows: {merged_rows:,}')
    print(f'Columns: {len(merged_df.columns):,}')
    print(f'Saved: {output_file}')

print('\n' + '=' * 80)
print(f'TOTAL ROWS ACROSS ALL MERGED APPLICATION CSVs: {grand_total_merged:,}')
print('=' * 80)

merge_summary_df = pd.DataFrame(merge_summary)
merge_summary_path = MERGED_DIR / 'merge_summary.csv'
merge_summary_df.to_csv(merge_summary_path, index=False)
print('Merge summary saved to:', merge_summary_path)


## Fast Row Count for Merged Application CSV Files


In [ ]:
from pathlib import Path
import pandas as pd

merged_csv_files = sorted([
    f for f in MERGED_DIR.glob('*.csv')
    if 'summary' not in f.name.lower()
])

print(f'Merged CSV files found: {len(merged_csv_files)}')

merged_summary = []
merged_total_rows = 0

if len(merged_csv_files) == 0:
    print('No merged CSV files found in:', MERGED_DIR)
else:
    print('=' * 80)
    print('FAST ROW COUNT REPORT - MERGED APPLICATION CSV FILES')
    print('=' * 80)

    for csv_file in merged_csv_files:
        with open(csv_file, 'r', encoding='utf-8', errors='ignore') as f:
            row_count = sum(1 for _ in f) - 1

        row_count = max(row_count, 0)
        merged_total_rows += row_count
        column_count = len(pd.read_csv(csv_file, nrows=0).columns)

        merged_summary.append({
            'file_name': csv_file.name,
            'rows': row_count,
            'columns': column_count
        })

        print(f'{csv_file.name:<45} {row_count:>12,} rows   {column_count:>5} columns')

    print('=' * 80)
    print(f'{"TOTAL MERGED ROWS":<45} {merged_total_rows:>12,}')
    print('=' * 80)

    merged_row_summary_df = pd.DataFrame(merged_summary, columns=['file_name', 'rows', 'columns'])
    merged_row_summary_df = pd.concat([
        merged_row_summary_df,
        pd.DataFrame([{'file_name': 'TOTAL', 'rows': merged_total_rows, 'columns': ''}])
    ], ignore_index=True)

    merged_row_summary_path = MERGED_DIR / 'merged_row_count_summary.csv'
    merged_row_summary_df.to_csv(merged_row_summary_path, index=False)
    print('Merged row-count summary saved to:', merged_row_summary_path)


## Preview One Merged CSV

Run this cell to inspect the first merged CSV file.


In [ ]:
import pandas as pd

merged_csv_files = sorted([
    f for f in MERGED_DIR.glob('*.csv')
    if 'summary' not in f.name.lower()
])

if len(merged_csv_files) > 0:
    sample_file = merged_csv_files[0]
    print('Previewing:', sample_file)
    sample_df = pd.read_csv(sample_file, nrows=5)
    display(sample_df)
    print('Columns:')
    for col in sample_df.columns:
        print(col)
else:
    print('No merged CSV files available to preview.')
